In [ ]:
# =====================================================================
# LLM 백엔드 핵심 정리 노트북
#   - 원본: FastAPI / OpenAI API 연동 / Streaming API / 실무형 API 구조 PDF
#   - 대상: REST·HTTP는 이미 아는 개발자 -> "LLM 특유의 부분" 위주로 압축
#
# 목차
#   PART 0. 환경 준비
#   PART 1. FastAPI 핵심 (Flask/Spring과 다른 점만)
#   PART 2. LLM API 호출 (requests 직접 호출 -> SDK)
#   PART 3. FastAPI 뒤에 LLM 두기 (/chat + 오류 처리)
#   PART 4. 스트리밍 (stream=True -> async generator -> StreamingResponse -> SSE)
#   PART 5. 스트리밍 안전 패턴 (DONE/EMPTY/ERROR, 타임아웃, 클라이언트 중단)
#   PART 6. 레이어드 구조 (Router / Service / Client / Schema / Config)
#   PART 7. 운영 확장 (프롬프트 템플릿, 로깅, 캐싱)
#
# 실행: 위에서부터 Shift+Enter. .env 에 BASE_URL / API_KEY / MODEL 이 있어야 LLM 셀이 동작함
# =====================================================================

In [ ]:
# =====================================================================
# PART 0. 환경 준비
# =====================================================================
# 필요한 패키지 (없으면 주석 해제 후 한 번 실행)
# %pip install fastapi uvicorn httpx openai python-dotenv requests

import os                                   # 환경변수 읽기
from getpass import getpass                 # 키를 화면에 안 보이게 입력받기
from dotenv import load_dotenv              # .env 파일 -> 환경변수로 로드

load_dotenv()                               # 같은 폴더의 .env 를 읽어 os.environ 에 넣음

BASE_URL = os.getenv("BASE_URL")            # OpenAI 호환 게이트웨이 주소 (예: MLAPI .../v1)
API_KEY  = os.getenv("API_KEY")             # 인증 키 -> 절대 코드에 하드코딩하지 않음
MODEL    = os.getenv("MODEL", "openai/gpt-4o-mini")  # 프로바이더 접두사 포함 모델명

if not API_KEY:                             # .env 에 없으면
    API_KEY = getpass("API_KEY: ")          # 셀 출력에 남지 않게 입력받기
if not BASE_URL:
    BASE_URL = input("BASE_URL: ")

print("BASE_URL:", BASE_URL)
print("API_KEY :", API_KEY[:4] + "..." if API_KEY else None)   # 키는 항상 마스킹해서 출력
print("MODEL   :", MODEL)

# [운영 규칙]
# - .env        : 실제 키. .gitignore 에 반드시 등록
# - .env.example: 키 이름만 적은 공유용 예시
# - 한 번이라도 커밋된 키는 유출된 것으로 보고 즉시 재발급

In [ ]:
# =====================================================================
# PART 1. FastAPI 핵심
# ---------------------------------------------------------------------
# Flask 와 비교해 새로 볼 것은 4가지뿐
#   1) 함수 인자 타입힌트 = 입력 파싱 + 검증 (Path / Query / Body 를 자동 구분)
#   2) Pydantic 모델 = Request/Response 스키마. 잘못된 입력은 함수 실행 전 422
#   3) response_model = 응답 모양 고정 + 의도치 않은 필드 노출 차단
#   4) /docs = 코드가 곧 Swagger 문서 (app.openapi() 로도 확인 가능)
# 422(형식 오류)는 Pydantic 이 자동, 400(비즈니스 규칙 위반)은 HTTPException 으로 직접
# =====================================================================
from fastapi import FastAPI, HTTPException          # 앱 객체, 직접 에러 응답
from pydantic import BaseModel, Field                # 스키마 정의, 필드 제약

app1 = FastAPI(title="FastAPI 핵심")                  # title 은 /docs 상단에 표시됨

class TextRequest(BaseModel):                        # 들어오는 Body 구조
    text: str                                        # 기본값 없음 -> 필수
    mode: str = "upper"                              # 기본값 있음 -> 선택

class TextResponse(BaseModel):                       # 나가는 Body 구조
    result: str

@app1.get("/health")                                 # GET /health
def health():
    return {"ok": True}                              # dict 반환 -> JSON 자동 직렬화

@app1.get("/items/{item_id}")                        # 경로 안의 {item_id} = Path Parameter
def read_item(item_id: int):                         # int 힌트 -> "abc" 면 자동 422
    return {"item_id": item_id}

@app1.get("/search")                                 # 경로에 없는 인자 = Query Parameter
def search(keyword: str, limit: int = 10):           # keyword 필수, limit 선택(기본 10)
    return {"keyword": keyword, "limit": limit}

def process_text(text: str, mode: str) -> str:       # 처리 로직은 엔드포인트 밖으로 분리 (-> 나중에 Service 층)
    if mode == "upper":
        return text.upper()
    if mode == "lower":
        return text.lower()
    if mode == "reverse":
        return text[::-1]
    raise ValueError(f"unsupported mode: {mode}")

@app1.post("/text/process", response_model=TextResponse)  # 응답 모양을 TextResponse 로 고정
def text_process(req: TextRequest):                  # BaseModel 타입 인자 = Request Body
    try:
        return TextResponse(result=process_text(req.text, req.mode))
    except ValueError as e:                          # 비즈니스 규칙 위반은
        raise HTTPException(status_code=400, detail=str(e))  # 직접 400 으로 변환

In [ ]:
# TestClient: 서버를 띄우지 않고 노트북 안에서 앱을 호출 (사용법은 requests 와 동일)
from fastapi.testclient import TestClient

t1 = TestClient(app1)                                          # 앱을 감싼 테스트 클라이언트

print(t1.get("/health").json())                                # 200 {'ok': True}
print(t1.get("/items/3").json())                               # 200 item_id=3
print(t1.get("/items/abc").status_code)                        # 422: int 변환 실패 (if 문 없이 자동)
print(t1.get("/search", params={"keyword": "llm"}).json())     # limit 생략 -> 기본값 10
print(t1.post("/text/process", json={"text": "Hello"}).json()) # mode 생략 -> upper
print(t1.post("/text/process", json={}).status_code)           # 422: 필수 필드 text 누락
print(t1.post("/text/process", json={"text": "a", "mode": "x"}).json())  # 400: 우리가 만든 에러
print(list(app1.openapi()["paths"].keys()))                    # /docs 의 원천 데이터 = 등록된 경로 목록

In [ ]:
# =====================================================================
# PART 2-1. requests 로 LLM API 직접 호출 (SDK 가 내부에서 하는 일)
# ---------------------------------------------------------------------
# - 스펙: OpenAI Chat Completions. MLAPI 같은 게이트웨이도 같은 스펙이라 BASE_URL 만 바꾸면 됨
# - 요청 = POST {BASE_URL}/chat/completions + Bearer 인증 + {model, messages, temperature}
# - 응답에서 쓰는 곳 = choices[0].message.content(답변), usage(토큰=비용)
# =====================================================================
import requests, json

endpoint = BASE_URL.rstrip("/") + "/chat/completions"   # 끝 슬래시 정리 후 경로 붙이기 (빠지면 404)

headers = {
    "Authorization": f"Bearer {API_KEY}",               # 인증 방식: Bearer 토큰 (틀리면 401)
    "Content-Type": "application/json",
}

payload = {
    "model": MODEL,                                      # 모델명 오타 -> 400/404
    "messages": [                                        # 대화 기록 리스트
        {"role": "system", "content": "한국어로 한 문장으로 답해."},   # 모델의 역할·규칙
        {"role": "user", "content": "FastAPI가 뭐야?"},               # 사용자 입력
    ],
    "temperature": 0.7,                                  # 낮을수록 일관, 높을수록 다양
}

res = requests.post(endpoint, json=payload, headers=headers, timeout=60)  # LLM 은 느림 -> timeout 필수
print("status:", res.status_code)                        # 가장 먼저 status 확인

if res.status_code == 200:
    data = res.json()                                    # JSON -> dict
    print("top keys:", list(data.keys()))                # 구조를 모르면 최상위 키부터 확인
    print("answer  :", data["choices"][0]["message"]["content"])   # 답변 텍스트
    print("finish  :", data["choices"][0]["finish_reason"])        # stop=정상 / length=잘림
    print("usage   :", data["usage"])                    # prompt/completion 토큰 수 = 과금 기준
else:
    print(res.text)                                      # 실패 시 본문에 원인이 있음

# 자주 보는 실패 신호
#   401 키 누락/오류 | 404 경로 오류(/chat/completions 누락) | 400 모델명·필드 오류 | 429 속도 제한 -> 잠시 후 재시도

In [ ]:
# =====================================================================
# PART 2-2. OpenAI SDK 로 같은 호출
# ---------------------------------------------------------------------
# - endpoint 조립 + 헤더 + JSON 파싱이 client 한 줄로 압축됨
# - 응답은 dict 가 아니라 ChatCompletion 객체 -> 점(.)으로 접근, model_dump() 로 dict 변환 가능
# =====================================================================
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)       # 접속·인증 정보를 객체에 캡슐화 (한 번만 만들고 재사용)

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "REST와 RPC 차이를 한 문장으로"}],
    temperature=0.3,
    max_tokens=30,                                         # 출력 토큰 상한. 작으면 답변이 잘림
)

print(resp.choices[0].message.content)                     # 답변
print(resp.choices[0].finish_reason)                       # max_tokens 에 걸리면 'length'  <- 200 인데도 "불완전"
print(resp.usage.prompt_tokens, resp.usage.completion_tokens)  # 입력/출력 토큰
# 실무 포인트: status 200 만 보고 성공 처리하면 안 됨. finish_reason == "length" 도 체크

In [ ]:
# =====================================================================
# PART 2-3. system 프롬프트 효과 + 멀티턴
# ---------------------------------------------------------------------
# - LLM API 는 stateless: 서버가 이전 대화를 기억하지 않음
# - 맥락 유지 = 이전 user/assistant 메시지를 매번 전부 다시 보냄
#   -> 대화가 길수록 요청이 커지고 비용·지연 증가 -> 컨텍스트 길이 관리가 설계 과제
# =====================================================================
def ask(messages, temperature=0.7):                        # 호출을 한 함수로 모아 두기
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
    return r.choices[0].message.content

# 같은 질문, system 만 다르게
q = {"role": "user", "content": "캐시가 뭐야?"}
print(ask([{"role": "system", "content": "초등학생에게 설명하듯 한 문장."}, q]))
print(ask([{"role": "system", "content": "시니어 개발자에게 한 문장, 전문용어 사용."}, q]))

# 멀티턴: 대화 기록을 누적해서 매번 전송
history = [{"role": "system", "content": "짧게 답해."}]    # 대화 기록 리스트
for user_msg in ["파이썬 웹 프레임워크 3개만 알려줘", "방금 말한 것 중 첫 번째는 언제 써?"]:
    history.append({"role": "user", "content": user_msg})           # 1) 사용자 발화 추가
    answer = ask(history)                                            # 2) 기록 전체를 보냄
    history.append({"role": "assistant", "content": answer})        # 3) 모델 답변도 기록에 추가
    print(f"Q: {user_msg}\nA: {answer}\n")
# 3)을 빼면 두 번째 질문의 "방금 말한 것"을 모델이 알 수 없음

In [ ]:
# =====================================================================
# PART 3. FastAPI 뒤에 LLM 두기: POST /chat
# ---------------------------------------------------------------------
# 왜 클라이언트가 LLM 을 직접 부르면 안 되나
#   - API Key 노출 / 사용량·비용 통제 불가 / 프롬프트·정책을 서버에서 강제 못 함
# 구조: 클라이언트 -> FastAPI /chat (키는 여기만) -> LLM
#
# 오류 처리 원칙
#   - LLM 은 외부 서비스라 언제든 실패 (키 만료, 429, 네트워크, 타임아웃, 모델명 오류)
#   - LLM 호출 줄만 try/except -> 의미 있는 HTTP 상태로 변환
#   - 원본 예외 메시지(str(e))는 클라이언트에 그대로 노출하지 않음 (내부 정보 유출)
# =====================================================================
import openai                                              # 예외 클래스 사용

app3 = FastAPI(title="LLM Chat API")

SYSTEM_PROMPT = "너는 친절한 백엔드 멘토야. 한국어로 3문장 이내로 답해."   # 페르소나는 서버에 고정

class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=2000)    # 빈 문자열·과도한 길이 차단 (422)
    temperature: float = Field(0.7, ge=0, le=2)             # 선택 + 범위 검증

class ChatResponse(BaseModel):                              # 원본 응답을 그대로 내리지 않고 필요한 것만
    reply: str
    model: str
    prompt_tokens: int
    completion_tokens: int

@app3.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    if not req.message.strip():                             # 공백만 있는 입력은 비즈니스 규칙으로 400
        raise HTTPException(400, "message is empty")
    try:
        r = client.chat.completions.create(                 # ← try 로 감싸는 건 외부 호출 이 줄
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": req.message},
            ],
            temperature=req.temperature,
            timeout=30,                                     # 요청 단위 타임아웃
        )
    except openai.RateLimitError:                           # 429: 상대가 바쁨 -> 그대로 429 (클라이언트 재시도 유도)
        raise HTTPException(429, "LLM rate limited, retry later")
    except openai.APITimeoutError:                          # 타임아웃 -> 504
        raise HTTPException(504, "LLM timeout")
    except openai.AuthenticationError:                      # 우리 서버 키 문제 -> 클라이언트 잘못 아님 -> 502
        raise HTTPException(502, "LLM auth failed")
    except openai.APIError:                                 # 그 외 LLM 측 실패 (모델명 오류, 5xx 등) -> 502
        raise HTTPException(502, "LLM call failed")
    return ChatResponse(
        reply=r.choices[0].message.content,
        model=r.model,
        prompt_tokens=r.usage.prompt_tokens,
        completion_tokens=r.usage.completion_tokens,
    )

# 상태 코드 역할 정리
#   422 = 형식 오류(Pydantic 자동) | 400 = 규칙 위반(우리가 판단) | 429/502/504 = LLM 쪽 실패를 변환

In [ ]:
t3 = TestClient(app3)
print(t3.post("/chat", json={"message": "FastAPI 장점?"}).json())       # 200 정상
print(t3.post("/chat", json={"message": 123}).status_code)              # 422 타입 오류
print(t3.post("/chat", json={"message": "   "}).status_code)            # 400 빈 메시지
print(t3.post("/chat", json={"message": "hi", "temperature": 5}).status_code)  # 422 범위 밖

In [ ]:
# =====================================================================
# PART 4-1. 스트리밍 기본: stream=True
# ---------------------------------------------------------------------
# - 일반 응답: 마지막 토큰까지 생성된 뒤 한 번에 도착 -> 첫 글자까지 수 초~수십 초
# - 스트리밍: 토큰이 생성되는 즉시 chunk 로 흘러옴 -> TTFT(첫 토큰까지 시간) 단축 = 체감 속도
# - 달라지는 것
#     반환 타입: ChatCompletion -> ChatCompletionChunk 의 iterator
#     내용 위치: message.content(전체) -> delta.content(이번에 추가된 조각)
#     완성본   : 직접 이어 붙여야 함 (로그·DB 저장용)
#     usage    : 기본으로 안 옴 -> stream_options 로 요청
# =====================================================================
import time

stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "스트리밍의 장점을 3줄로"}],
    stream=True,                                           # 핵심 옵션 한 줄
    stream_options={"include_usage": True},                # 마지막에 usage 만 담긴 chunk 를 추가로 받음 (과금·모니터링 필수)
)

collected = []                                             # 조각 누적용
usage = None
t0 = time.perf_counter()
ttft = None
for chunk in stream:                                       # chunk 가 도착할 때마다 한 바퀴
    if not chunk.choices:                                  # 가드1: usage 전용 마지막 chunk 는 choices 가 비어 있음
        usage = chunk.usage                                #        -> 여기서 토큰 사용량 회수
        continue
    piece = chunk.choices[0].delta.content                 # 이번 조각 (첫 chunk 는 role 만 있고 content=None)
    if piece:                                              # 가드2: None 과 "" 를 한 번에 거름
        if ttft is None:
            ttft = time.perf_counter() - t0                # 첫 토큰 도착 시각 = TTFT
        print(piece, end="", flush=True)                   # 줄바꿈·버퍼링 없이 즉시 출력
        collected.append(piece)                            # 동시에 누적

full_text = "".join(collected)                             # 리스트에 모았다가 join (문자열 += 반복보다 효율적)
print(f"\n\nTTFT={ttft:.2f}s  total={time.perf_counter()-t0:.2f}s  len={len(full_text)}")
print("usage:", usage)

In [ ]:
# =====================================================================
# PART 4-2. async generator (스트리밍 서버의 뼈대)
# ---------------------------------------------------------------------
# - generator      : def + yield. 값을 한 번에 만들지 않고 하나씩 흘림
# - async generator: async def + yield. yield 사이에 await 가능 (= 네트워크 대기 중 다른 요청 처리)
# - 소비는 async for. 변환 generator 가 원본을 소비해 가공 후 재방출(yield)하는 파이프라인 패턴
# =====================================================================
import asyncio

async def fake_tokens():                                   # LLM 토큰 스트림 흉내
    for tok in ["Fast", "", "API", " ", None, "is", " ", "fast"]:
        await asyncio.sleep(0.1)                           # 네트워크 지연 흉내 (여기서 이벤트 루프가 다른 일 가능)
        yield tok                                          # 한 조각 내보내고 멈춤

async def clean_upper(source):                             # 변환 generator: 원본을 받아 가공
    async for tok in source:                               # 원본 소비
        if not tok:                                        # 빈 chunk 가드는 변환 단계에 모아 둠
            continue
        yield tok.upper()                                  # 가공 후 재방출

async for tok in clean_upper(fake_tokens()):               # Jupyter 는 최상위 async for/await 허용
    print(repr(tok))

In [ ]:
# =====================================================================
# PART 4-3. FastAPI StreamingResponse
# ---------------------------------------------------------------------
# - StreamingResponse(generator) 한 줄이면 응답 본문이 chunk 단위로 흐름
# - source 가 더미든 LLM 이든 StreamingResponse 입장에서는 동일
# - 받는 쪽: httpx 의 stream() + iter_text()
# =====================================================================
from fastapi.responses import StreamingResponse
from openai import AsyncOpenAI

aclient = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)  # 스트리밍 서버는 Async 클라이언트로 통일 (이유는 PART 5)

app4 = FastAPI()

@app4.get("/dummy/stream")
async def dummy_stream():
    async def gen():
        for i in range(5):
            await asyncio.sleep(0.2)
            yield f"line {i}\n"                            # yield 할 때마다 클라이언트로 전송
    return StreamingResponse(gen(), media_type="text/plain")

@app4.post("/chat/stream")
async def chat_stream(req: ChatRequest):                   # 입력 검증은 일반 API 와 동일하게 Pydantic
    async def gen():
        stream = await aclient.chat.completions.create(    # Async 클라이언트는 await 필요
            model=MODEL,
            messages=[{"role": "user", "content": req.message}],
            stream=True,
        )
        async for chunk in stream:                         # 4-1 의 루프를 async 로
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece                                # 가공이 필요하면 이 한 줄이 그 자리
    return StreamingResponse(gen(), media_type="text/plain")

# 스트리밍은 실제 서버로 확인해야 정확함 (TestClient 는 본문을 모아서 줄 수 있음)
# 노트북에서 uvicorn.run() 을 그냥 부르면 Jupyter 이벤트 루프와 충돌 -> 백그라운드 스레드로 실행
import threading, uvicorn, httpx

def run_in_thread(app, port):
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()   # 별도 스레드 = 별도 이벤트 루프
    time.sleep(1.5)                                        # 기동 대기
    return server                                          # 종료: server.should_exit = True

server4 = run_in_thread(app4, 8764)                        # 포트 충돌 시 번호만 변경
S4 = "http://127.0.0.1:8764"

t0 = time.perf_counter()
with httpx.Client(timeout=30) as hc:
    with hc.stream("GET", f"{S4}/dummy/stream") as r:     # 응답을 한 번에 읽지 않고 스트림으로 열기
        for text in r.iter_text():                         # 도착하는 대로 받기
            print(f"{time.perf_counter()-t0:.2f}s", repr(text))   # 0.2초 간격이면 스트리밍이 살아 있음

    with hc.stream("POST", f"{S4}/chat/stream", json={"message": "SSE를 한 문장으로"}) as r:
        for text in r.iter_text():
            print(text, end="", flush=True)

In [ ]:
# =====================================================================
# PART 4-4. SSE (Server-Sent Events) 형식
# ---------------------------------------------------------------------
# - 서버 -> 클라이언트 단방향. 일반 HTTP 응답을 끊지 않고 이벤트를 계속 보냄
# - LLM 챗은 "질문 1번 -> 답변 길게"라 단방향으로 충분. WebSocket 보다 운영이 단순
#   (기존 인증 헤더·로드밸런서·프록시 그대로, 무상태라 수평 확장 쉬움)
# - text/plain 스트림과의 차이: 이벤트 경계와 종류를 표현할 수 있음
#     프레임 = "event: 이름\n" (선택) + "data: 내용\n" + 빈 줄("\n")
# - 브라우저 EventSource 는 GET 만 되고 커스텀 헤더 불가
#   -> 실무는 fetch(POST) + response.body.getReader() 로 받고 data: 를 직접 파싱
#      (또는 @microsoft/fetch-event-source 라이브러리)
# - Nginx 등 프록시가 버퍼링하면 스트리밍이 깨짐 -> X-Accel-Buffering: no
# =====================================================================
def sse(data: str, event: str | None = None) -> str:      # SSE 프레임 한 개 만들기
    data = json.dumps(data, ensure_ascii=False)            # 조각 안의 줄바꿈이 프레임을 깨지 않게 JSON 문자열로
    head = f"event: {event}\n" if event else ""            # 이벤트 종류 (없으면 기본 message)
    return f"{head}data: {data}\n\n"                       # 빈 줄로 프레임 끝을 표시

@app4.post("/chat/sse")
async def chat_sse(req: ChatRequest):
    async def gen():
        stream = await aclient.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": req.message}], stream=True,
        )
        async for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                yield sse(chunk.choices[0].delta.content)  # 토큰 이벤트
        yield sse("[DONE]", event="done")                  # 종료 이벤트
    return StreamingResponse(
        gen(),
        media_type="text/event-stream",                    # SSE 의 Content-Type
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},  # 캐시·프록시 버퍼링 방지
    )

# 받는 쪽: 줄 단위로 읽어 "data: " 를 파싱 (브라우저 fetch 에서 하는 일과 동일)
with httpx.Client(timeout=30) as hc, hc.stream("POST", f"{S4}/chat/sse", json={"message": "토큰이 뭐야? 한 문장"}) as r:
    event = "message"
    for line in r.iter_lines():                            # SSE 는 줄 기반 프로토콜
        if line.startswith("event: "):
            event = line[7:]                               # 다음 data 의 이벤트 종류
        elif line.startswith("data: "):
            payload_ = json.loads(line[6:])
            if event == "done":
                print("\n[종료]")
            else:
                print(payload_, end="", flush=True)
        elif line == "":                                   # 빈 줄 = 프레임 끝 -> 이벤트 종류 초기화
            event = "message"

In [ ]:
# =====================================================================
# PART 5. 스트리밍 안전 패턴 (실무 핵심)
# ---------------------------------------------------------------------
# 스트리밍은 일반 응답과 "깨지는 방식"이 다름
#   1) 도중 오류 : 첫 조각을 보낸 순간 이미 200 전송 완료 -> status 로 실패를 알릴 수 없음
#                  => 본문 마지막 줄에 [ERROR] 마커
#   2) 빈 응답   : 호출은 성공인데 yield 가 0번 -> 200 + 본문 0바이트 (실패와 구분 불가)
#                  => produced 플래그로 추적해 [EMPTY] 마커
#   3) 타임아웃 : 서버(LLM 호출 timeout) + 클라이언트(httpx timeout) 두 겹으로
#   4) 클라이언트 중단: 사용자가 창을 닫으면 CancelledError 가 generator 로 전파
#                  => 로그 남기고 반드시 re-raise (삼키면 FastAPI 가 정상 종료로 오해)
#                  => finally 에서 리소스 정리 (중단 안 하면 아무도 안 보는 답변에 토큰 비용 계속 발생)
#   마지막 줄을 [DONE] / [EMPTY] / [ERROR] 중 하나로 고정 -> 받는 쪽이 한 줄로 종료 사유 판단
#
# 왜 AsyncOpenAI + async def 로 통일?
#   sync 클라이언트는 블로킹 -> 연결이 끊겨도 CancelledError 가 늦게 오거나 안 옴
# =====================================================================
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
for _n in ("httpx", "httpx2"):
    logging.getLogger(_n).setLevel(logging.WARNING)      # 요청마다 찍히는 HTTP 클라이언트 로그 끄기
log = logging.getLogger("chat")

app5 = FastAPI()

class StreamRequest(BaseModel):
    message: str = Field(min_length=1)
    model: str | None = None                               # 선택: 잘못된 모델명 테스트용
    temperature: float = Field(0.7, ge=0, le=2)

async def stream_chat_safe(req: StreamRequest):
    produced = False                                       # 한 번이라도 yield 했는지
    try:
        stream = await aclient.chat.completions.create(
            model=req.model or MODEL,
            messages=[{"role": "user", "content": req.message}],
            temperature=req.temperature,
            stream=True,
            timeout=30,                                    # 서버 쪽 타임아웃 (무한 대기 방지)
        )
        async for chunk in stream:
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                produced = True
                yield piece
        yield "\n[DONE]" if produced else "\n[EMPTY]"      # 정상 종료 vs 성공했지만 결과 없음
    except asyncio.CancelledError:                         # 클라이언트가 연결을 끊음
        log.info("cancelled by client")                    # 기록만 하고
        raise                                              # 반드시 다시 던짐
    except Exception as e:                                 # LLM 호출 실패 (이미 200 이 나갔을 수 있음)
        log.error("stream failed: %r", e)                  # 상세는 서버 로그에만 (운영에선 log.exception 으로 스택까지)
        yield "\n[ERROR] llm call failed"                  # 클라이언트엔 마커만 (원본 메시지 노출 X)
    finally:
        log.info("generator finished or cancelled")        # 정상·오류·중단 모든 경로에서 실행 -> 정리 코드 자리

@app5.post("/chat/stream/safe")
async def chat_stream_safe(req: StreamRequest):
    return StreamingResponse(stream_chat_safe(req), media_type="text/plain")

In [ ]:
# 클라이언트 중단(CancelledError)은 실제 서버에서만 재현됨 -> 4-3 의 run_in_thread 재사용
server5 = run_in_thread(app5, 8765)
SERVER = "http://127.0.0.1:8765"

In [ ]:
def call(body, max_chunks=None, timeout=60.0):
    with httpx.Client(timeout=timeout) as c:               # 클라이언트 쪽 타임아웃 (두 번째 방어선)
        with c.stream("POST", f"{SERVER}/chat/stream/safe", json=body) as r:
            print("status:", r.status_code)                # 스트리밍은 실패해도 200 일 수 있음
            text = ""
            for i, piece in enumerate(r.iter_text()):
                text += piece
                if max_chunks and i + 1 >= max_chunks:
                    print("클라이언트가 중단(break)")
                    break                                  # 연결 종료 -> 서버 generator 에 CancelledError
            print("마지막 줄:", text.strip().splitlines()[-1] if text.strip() else "(본문 없음)")

call({"message": "HTTP/2 장점 두 문장"})                   # 기대: 200 + [DONE]
call({"message": "hi", "model": "this-model-does-not-exist"})  # 기대: 200 + [ERROR], 서버는 계속 살아있음
call({"message": "파이썬 역사를 길게 설명해"}, max_chunks=3)  # 기대: 서버 로그에 cancelled by client -> finished
time.sleep(1)                                              # 서버 로그 출력 대기
# 검증 포인트: status 200 만 보지 말고 "마지막 줄 마커 + 서버 로그"까지 함께 확인

In [ ]:
# =====================================================================
# PART 6. 레이어드 구조
# ---------------------------------------------------------------------
# 한 함수에 몰려 있던 코드를 책임별로 분리. 의존성은 한 방향: Router -> Service -> Client
#   core/config.py    : .env 값을 한 곳에서 읽음 (키 없으면 기동 시점에 RuntimeError)
#   schemas/chat.py   : Pydantic 요청/응답 모델 (Router 와 Service 가 공유)
#   clients/llm.py    : LLM SDK 를 아는 유일한 곳 -> 공급자/SDK 가 바뀌면 여기만 수정
#   services/chat.py  : 비즈니스 로직 (메시지 조립, 프롬프트, 안전 패턴). HTTP 를 모름
#   routers/chat.py   : HTTP 입출력만 (요청 받기 -> Service 위임 -> 응답 형태 결정)
#   main.py           : 앱 생성 + 라우터 등록만
# 이득: 프롬프트 변경=Service, 호출 방식 변경=Client, URL·응답 모양=Router 한 곳만 수정
#       Client 를 가짜로 바꿔 끼우면 LLM 없이 Service/Router 테스트 가능 (LLM 응답은 매번 달라서 중요)
# 아래 셀은 파일만 생성함
# =====================================================================
from pathlib import Path

files = {}

files["app/core/config.py"] = """
import os
from dotenv import load_dotenv

load_dotenv()                                   # .env -> 환경변수

BASE_URL = os.getenv("BASE_URL")
API_KEY = os.getenv("API_KEY")
MODEL = os.getenv("MODEL", "openai/gpt-4o-mini")

if not API_KEY:                                 # 설정 누락은 요청 때가 아니라 기동 시점에 바로 실패
    raise RuntimeError("API_KEY is not set")
"""

files["app/schemas/chat.py"] = """
from pydantic import BaseModel, Field

class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=2000)
    model: str | None = None
    temperature: float = Field(0.7, ge=0, le=2)

class ChatResponse(BaseModel):                  # 비스트리밍 응답용
    reply: str
"""

files["app/clients/llm.py"] = """
from openai import AsyncOpenAI
from app.core import config

_client = AsyncOpenAI(base_url=config.BASE_URL, api_key=config.API_KEY)   # 모듈 레벨에서 한 번만 생성

async def stream_chat(messages, model=None, temperature=0.7):
    # 메시지 리스트 -> 토큰 문자열 async generator (SDK 의 chunk 구조는 여기 밖으로 새지 않음)
    stream = await _client.chat.completions.create(
        model=model or config.MODEL, messages=messages,
        temperature=temperature, stream=True, timeout=30,
    )
    async for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content

async def complete_chat(messages, model=None, temperature=0.7) -> str:
    # 비스트리밍 단발 호출
    r = await _client.chat.completions.create(
        model=model or config.MODEL, messages=messages,
        temperature=temperature, timeout=30,
    )
    return r.choices[0].message.content
"""

files["app/services/chat.py"] = """
import asyncio, logging
from app.clients import llm
from app.schemas.chat import ChatRequest

log = logging.getLogger("chat_service")
SYSTEM_PROMPT = "너는 친절한 백엔드 멘토야. 한국어로 간결하게 답해."   # 프롬프트 정책은 Service 소관

def build_messages(req: ChatRequest):
    # 요청 객체 -> LLM 메시지 리스트 (대화 기록·RAG 문맥을 붙인다면 여기)
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": req.message}]

async def stream_chat_safe(req: ChatRequest):
    # PART 5 의 안전 패턴이 그대로 이사 옴. HTTP 개념(status, Response)은 전혀 모름
    produced = False
    try:
        async for piece in llm.stream_chat(build_messages(req), req.model, req.temperature):
            produced = True
            yield piece
        yield "\\n[DONE]" if produced else "\\n[EMPTY]"
    except asyncio.CancelledError:
        log.info("chat service cancelled by client")
        raise
    except Exception as e:
        log.error("chat service failed: %r", e)
        yield "\\n[ERROR] llm call failed"
    finally:
        log.info("chat service generator finished or cancelled")

async def chat_once(req: ChatRequest) -> str:
    return await llm.complete_chat(build_messages(req), req.model, req.temperature)
"""

files["app/routers/chat.py"] = """
from fastapi import APIRouter, HTTPException
from fastapi.responses import StreamingResponse
from app.schemas.chat import ChatRequest, ChatResponse
from app.services import chat as chat_service

router = APIRouter(prefix="/chat", tags=["chat"])   # 경로 묶음

@router.post("/stream")
async def chat_stream(req: ChatRequest):
    # 로직 없음: Service 의 async generator 를 StreamingResponse 에 넘기는 한 줄 위임
    return StreamingResponse(chat_service.stream_chat_safe(req), media_type="text/plain")

@router.post("", response_model=ChatResponse)
async def chat(req: ChatRequest):
    try:
        return ChatResponse(reply=await chat_service.chat_once(req))
    except Exception:
        raise HTTPException(502, "LLM call failed")   # 예외 -> HTTP 상태 변환은 Router 소관
"""

files["app/main.py"] = """
from fastapi import FastAPI
from app.routers import chat

app = FastAPI(title="LLM Backend")
app.include_router(chat.router)                     # 라우터 등록

@app.get("/health")
def health():
    return {"ok": True}
"""

for path, code in files.items():
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)     # 폴더 생성
    for parent in [p.parent, *p.parent.parents][:-1]:
        (parent / "__init__.py").touch()            # 패키지로 인식되게 __init__.py 생성
    p.write_text(code.lstrip(), encoding="utf-8")   # 파일 쓰기 (다시 실행해도 안전하게 덮어씀)

print("\n".join(sorted(str(p) for p in Path("app").rglob("*.py"))))
# 서버 실행(터미널): uvicorn app.main:app --reload --port 8000

In [ ]:
# 분리 후에도 동작이 같은지 검증 (TestClient 로 import 해서 바로 호출)
import sys, importlib
sys.path.insert(0, os.getcwd())                             # 현재 폴더를 import 경로에
import app.main as app_main
importlib.reload(app_main)

# with 블록으로 써야 요청들이 같은 이벤트 루프를 공유함
# (모듈 레벨 AsyncOpenAI 의 커넥션 풀이 첫 요청의 루프에 묶이기 때문. 실제 uvicorn 은 루프 하나라 문제없음)
with TestClient(app_main.app) as t6:
    print(t6.get("/health").json())
    print(t6.post("/chat", json={"message": "레이어드 구조 장점 한 문장"}).json())
    with t6.stream("POST", "/chat/stream", json={"message": "의존성 역전 한 문장"}) as r:
        print("".join(r.iter_text()))                      # 마지막 줄 [DONE] 확인

In [ ]:
# =====================================================================
# PART 7. 운영 확장: 프롬프트 템플릿 / 로깅 / 캐싱
# ---------------------------------------------------------------------
# 프롬프트
#   - 결과를 좌우하는 건 호출 코드보다 프롬프트. 코드에 문자열로 흩어두면 변경·비교·롤백이 어려움
#   - system 권장 순서: 역할 -> 톤 -> 언어 -> 제약
#   - 템플릿 = 변수 자리만 비운 틀. 보관: 코드 상수 -> 파일(yaml 등) -> DB/프롬프트 관리 도구 순으로 성장
# 로깅 (print 대신 logging: 레벨·포맷·출력처 제어)
#   - 남길 것: 요청 id, 모델, 입력/출력 토큰, 전체 지연, TTFT, 종료 사유(DONE/EMPTY/ERROR/cancel)
#   - 원문 프롬프트·개인정보·키는 남기지 않거나 마스킹
# 캐싱
#   - LLM 호출은 느리고 비쌈 -> 같은 입력 반복이 많은 곳(FAQ, 분류, 요약)에 효과적
#   - 캐시 키 = 결과에 영향을 주는 모든 것 (model + messages + temperature ...)
#   - temperature 높은 창작형·개인화 대화에는 부적합
#   - 저장소: 프로세스 메모리(dict/LRU) -> 여러 서버면 Redis
# =====================================================================
import hashlib, uuid

PROMPTS = {                                                # 프롬프트를 이름으로 관리 (코드와 분리하는 첫 단계)
    "summarize": "너는 {domain} 전문가야. 다음 글을 {n}문장으로 요약해. 한국어로.",
}

def render(name: str, **vars) -> str:
    return PROMPTS[name].format(**vars)                    # 변수 자리 채우기

_cache: dict[str, str] = {}                                # 데모용 메모리 캐시

def cache_key(model, messages, temperature) -> str:
    raw = json.dumps({"m": model, "msg": messages, "t": temperature}, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode()).hexdigest()        # 입력 전체를 해시 -> 고정 길이 키

def chat_with_cache_and_log(messages, temperature=0.0):
    rid = uuid.uuid4().hex[:8]                             # 요청 추적 id (로그 묶기용)
    key = cache_key(MODEL, messages, temperature)
    if key in _cache:
        log.info("rid=%s cache=hit", rid)
        return _cache[key]
    t0 = time.perf_counter()
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
    answer = r.choices[0].message.content
    log.info("rid=%s cache=miss model=%s in=%d out=%d finish=%s latency=%.2fs",
             rid, r.model, r.usage.prompt_tokens, r.usage.completion_tokens,
             r.choices[0].finish_reason, time.perf_counter() - t0)   # 원문 대신 메타데이터만
    _cache[key] = answer
    return answer

msgs = [{"role": "system", "content": render("summarize", domain="백엔드", n=1)},
        {"role": "user", "content": "SSE는 서버가 클라이언트로 HTTP 연결을 유지한 채 이벤트를 계속 보내는 방식이다."}]
print(chat_with_cache_and_log(msgs))                       # miss -> LLM 호출
print(chat_with_cache_and_log(msgs))                       # hit  -> 즉시 반환, 비용 0

In [ ]:
# =====================================================================
# 마무리 체크리스트 (자료 없이 직접 만들 수 있으면 완료)
# ---------------------------------------------------------------------
# [ ] .env / .env.example / .gitignore 로 키 분리, 키는 서버에만
# [ ] POST /chat: Pydantic 검증(422) + 규칙 위반(400) + LLM 실패 변환(429/502/504)
# [ ] finish_reason == "length" 처리 (200 인데 잘린 답변)
# [ ] 멀티턴: 기록 누적 전송 + 길이 관리
# [ ] POST /chat/stream: AsyncOpenAI + async generator + StreamingResponse
# [ ] 가드: if not chunk.choices / if piece
# [ ] 마지막 줄 [DONE]/[EMPTY]/[ERROR], CancelledError re-raise, finally 정리, 타임아웃 두 겹
# [ ] SSE 형식(text/event-stream, data: ...\n\n) + 프록시 버퍼링 끄기
# [ ] Router / Service / Client / Schema / Config 분리
# [ ] 로그: 요청 id, 토큰, 지연, TTFT, 종료 사유
#
# 자료에 없지만 다음에 볼 것
#   - 429 재시도(exponential backoff), 대화 기록 DB 저장 시점(스트림 완료/중단), 인증(JWT),
#     Client mock 으로 테스트, 배포(Docker) 시 프록시 버퍼링·타임아웃 설정
# =====================================================================
server4.should_exit = True                                 # 백그라운드 서버 종료
server5.should_exit = True